# Audiobooks Customer Behavior Prediction
Author: Tuncay Sahin  
Project: Deep Learning Foundations with TensorFlow 2  
Focus: Customer Behavior Prediction
## Environment Verification

This notebook runs on:

- TensorFlow 2.20.0
- Python Environment: py3-TF2.0

---

# Problem Understanding

## Problem Tanımı

Bu çalışmanın amacı, audiobook platformunu kullanan müşterilerin gelecekte tekrar satın alma (**repeat purchase**) davranışlarını tahmin etmektir.

Makine öğrenmesi yaklaşımı kullanılarak müşterilerin geçmiş kullanım alışkanlıkları analiz edilmekte ve tekrar satın alma olasılığı modellenmektedir.

---

## İş Problemi (Business Perspective)

Dijital platformlarda yeni müşteri kazanımı, mevcut müşteriyi elde tutmaya kıyasla daha maliyetlidir.

Bu nedenle temel iş sorusu şudur:

> Hangi müşteriler tekrar satın alma potansiyeline sahiptir?

Bu tahmin sayesinde:

- Pazarlama kampanyaları doğru müşteri segmentlerine yönlendirilebilir
- Gereksiz reklam maliyetleri azaltılabilir
- Müşteri yaşam boyu değeri artırılabilir

---

## Problem Türü

Bu çalışma bir **Binary Classification Problem** olarak ele alınmaktadır.

Hedef değişken:

- 0 → Tek seferlik müşteri
- 1 → Tekrar satın alan müşteri

---

## Veri Seti Genel Yapısı

Veri seti müşteri davranışlarını temsil eden sayısal özelliklerden oluşmaktadır.

Toplam gözlem sayısı:

**14,084 müşteri**

Model girdileri aşağıdaki davranışsal metrikleri temsil etmektedir:

- Satın alma sıklığı
- Dinleme süresi
- Platform kullanım davranışı
- Geçmiş kullanıcı etkileşim metrikleri

---

## Problemdeki Temel Zorluk

Veri seti incelendiğinde sınıf dağılımının dengeli olmadığı görülmektedir.

- Majority Class → Non Repeat Customers
- Minority Class → Repeat Customers

Bu durum modelin yalnızca çoğunluk sınıfını öğrenmesine neden olabilir.

Bu nedenle model geliştirme sürecinde:

- Class imbalance analizi
- Threshold optimizasyonu

uygulanacaktır.

---

## Modelleme Amacı

Bu çalışmada amaç yalnızca yüksek doğruluk (**accuracy**) elde etmek değildir.

Asıl hedef:

- Repeat customer tahmininde hata oranını azaltmak
- Model kararlarını iş hedefleri ile uyumlu hale getirmektir.

Bu doğrultuda model geliştirme sürecinde:

- Class imbalance handling
- ROC eğrisi değerlendirmesi
- Classification threshold optimizasyonu

uygulanmıştır.

---

## Çalışma Akışı

Proje aşağıdaki aşamalardan oluşmaktadır:

1. Problem Tanımı
2. Veri Ön İşleme
3. Model Eğitimi
4. Model Performans Analizi
5. Karar Eşiği (Threshold) Optimizasyonu
6. Nihai Modelin Kaydedilmesi

In [1]:
from pathlib import Path
import pandas as pd

# Bu notebook dosyasının bulunduğu klasör (Jupyter'de bazen cwd sapabilir)
# O yüzden kökü elle tespit edeceğiz:
project_root = Path.cwd()

# Eğer şu an notebooks içindeysen, bir üst klasör proje kökü olur
# Eğer proje kökündeysen, olduğu gibi kalır
if (project_root / "data").exists() is False and (project_root.parent / "data").exists():
    project_root = project_root.parent

data_path = project_root / "data" / "Audiobooks_data.csv"

df = pd.read_csv(data_path, header=None)
df.head()

,0,1,2,3,4,5,6,7,8,9,10,11
0,994,1620.0,1620,19.73,19.73,1,10.00,0.99,1603.8,5,92,0
1,1143,2160.0,2160,5.33,5.33,0,8.91,0.00,0.0,0,0,0
2,2059,2160.0,2160,5.33,5.33,0,8.91,0.00,0.0,0,388,0
3,2882,1620.0,1620,5.96,5.96,0,8.91,0.42,680.4,1,129,0
4,3342,2160.0,2160,5.33,5.33,0,8.91,0.22,475.2,0,361,0


In [2]:
import tensorflow as tf

tf.__version__

'2.20.0'

In [3]:
import pandas as pd

DATA_PATH = "../data/Audiobooks_data.csv"

df = pd.read_csv(DATA_PATH, header=None)

print("Dataset Shape:", df.shape)

df.head()

Dataset Shape: (14084, 12)


,0,1,2,3,4,5,6,7,8,9,10,11
0,994,1620.0,1620,19.73,19.73,1,10.00,0.99,1603.8,5,92,0
1,1143,2160.0,2160,5.33,5.33,0,8.91,0.00,0.0,0,0,0
2,2059,2160.0,2160,5.33,5.33,0,8.91,0.00,0.0,0,388,0
3,2882,1620.0,1620,5.96,5.96,0,8.91,0.42,680.4,1,129,0
4,3342,2160.0,2160,5.33,5.33,0,8.91,0.22,475.2,0,361,0


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 14084 entries, 0 to 14083
Data columns (total 12 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   0       14084 non-null  int64  
 1   1       14084 non-null  float64
 2   2       14084 non-null  int64  
 3   3       14084 non-null  float64
 4   4       14084 non-null  float64
 5   5       14084 non-null  int64  
 6   6       14084 non-null  float64
 7   7       14084 non-null  float64
 8   8       14084 non-null  float64
 9   9       14084 non-null  int64  
 10  10      14084 non-null  int64  
 11  11      14084 non-null  int64  
dtypes: float64(6), int64(6)
memory usage: 1.3 MB


### Veri Seti Yapısı İncelemesi

Veri seti toplam **14.084 gözlem** ve **12 sayısal değişkenden** oluşmaktadır.

Kolonlar incelendiğinde:

- Eksik veri (missing value) bulunmamaktadır.
- Değişken tipleri tamamen sayısaldır (int ve float).
- Veri seti doğrudan makine öğrenmesi modelleri için kullanılabilir durumdadır.

Bu durum ek bir eksik veri temizleme sürecine ihtiyaç olmadığını göstermektedir.

In [5]:
df.describe()

,0,1,2,3,4,5,6,7,8,9,10,11
count,14084.000000,14084.000000,14084.000000,14084.000000,14084.000000,14084.000000,14084.000000,14084.000000,14084.000000,14084.000000,14084.000000,14084.000000
mean,16772.491551,1591.281685,1678.608634,7.103791,7.543805,0.160750,8.909795,0.125659,189.888983,0.070222,61.935033,0.158833
std,9691.807248,504.340663,654.838599,4.931673,5.560129,0.367313,0.643406,0.241206,371.084010,0.472157,88.207634,0.365533
min,2.000000,216.000000,216.000000,3.860000,3.860000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,8368.000000,1188.000000,1188.000000,5.330000,5.330000,0.000000,8.910000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,16711.500000,1620.000000,1620.000000,5.950000,6.070000,0.000000,8.910000,0.000000,0.000000,0.000000,11.000000,0.000000
75%,25187.250000,2160.000000,2160.000000,8.000000,8.000000,0.000000,8.910000,0.130000,194.400000,0.000000,105.000000,0.000000
max,33683.000000,2160.000000,7020.000000,130.940000,130.940000,1.000000,10.000000,1.000000,2160.000000,30.000000,464.000000,1.000000


In [6]:
df[11].value_counts()

11
0    11847
1     2237
Name: count, dtype: int64

### Hedef Değişken Dağılımı (Class Imbalance)

Hedef değişken incelendiğinde veri setinin dengeli olmadığı görülmektedir.

- 0 → Tek seferlik müşteri (çoğunluk sınıf)
- 1 → Tekrar satın alan müşteri (azınlık sınıf)

Bu durum modelin yalnızca çoğunluk sınıfı öğrenmesine neden olabilir.

Bu nedenle model geliştirme sürecinde:

- Precision / Recall dengesi incelenecek
- ROC eğrisi değerlendirilecek
- Classification threshold optimizasyonu uygulanacaktır.

In [7]:
df[0].nunique(), len(df)

(14084, 14084)

### Müşteri Kimlik (Identifier) Kontrolü

İlk kolon incelendiğinde her gözlem için benzersüz (unique) değer içerdiği görülmektedir.

Bu kolon müşteri kimliği (ID) niteliğindedir ve tahmin gücü taşımaz.

Bu nedenle model eğitim sürecinde özellik (feature) olarak kullanılmamıştır.

In [8]:
# Feature ve Target ayrımı

FEATURES = df.iloc[:, 1:-1]
TARGET = df.iloc[:, -1]

X = FEATURES.values
y = TARGET.values

print("Feature Shape:", X.shape)
print("Target Shape:", y.shape)

Feature Shape: (14084, 10)
Target Shape: (14084,)


### Model Girdi ve Hedef Değişken Ayrımı

Model eğitimi için veri iki ana bileşene ayrılmıştır:

- **X (Features):** Müşteri davranışlarını temsil eden sayısal değişkenler
- **y (Target):** Müşterinin tekrar satın alma durumu

Toplamda:

- 10 adet model girdisi
- 1 adet hedef değişken bulunmaktadır.

Kimlik (ID) kolonu model öğrenmesine katkı sağlamadığı için özellik setine dahil edilmemiştir.

Bu yapı Binary Classification problemi için hazırlanmıştır.

## Notebook Özeti

Bu aşamada veri seti detaylı olarak analiz edilmiştir:

- Veri başarıyla yüklenmiştir
- Veri tipleri ve yapısı incelenmiştir
- Eksik veri kontrolü gerçekleştirilmiştir
- Hedef değişken dağılımı analiz edilmiştir
- Kimlik (ID) kolonu tespit edilerek model dışında bırakılmıştır
- Model eğitiminde kullanılacak girdiler hazırlanmıştır

Bu analiz sonucunda veri setinin modelleme aşamasına hazır olduğu doğrulanmıştır.

Bir sonraki aşamada veri ön işleme (preprocessing), ölçekleme ve veri bölme işlemleri gerçekleştirilecektir.